
# PCA research notebook for hedge design and model selection

This notebook is built to study **which PCA specification is the most useful for the desk**, not just which one gives the highest explained variance.

The notebook is organized around five questions:

1. **How should the data be preprocessed?**
   - levels vs changes
   - demeaned vs standardized
2. **Which covariance estimator is the most robust?**
   - sample
   - Ledoit-Wolf
   - OAS
   - EWMA
   - Graphical Lasso variants if desired
3. **How many PCs should be retained?**
4. **Which hedge tenors should represent the PCA hedge?**
5. **Which models remain stable through time and in stressed regimes?**

The point is not to automate the decision blindly. The point is to create a **complete diagnostic framework** so that you can compare models on:

- explained variance
- reconstruction quality
- hedge PNL replication quality
- parameter stability through time
- robustness by market regime
- condition number / practical hedge implementability

---

## PNL methodology implemented

For a holding horizon of `h` business-day rows, the notebook evaluates:

\[
	ext{PCA Move}_t = P_{t+h} - P_t
\]

\[
	ext{PCA PNL}_t = \sum_{j \in 	ext{hedge tenors}} 	ext{PCA Move}_{t,j} 	imes 	ext{PCA Sensi}_j
\]

\[
	ext{True PNL}_t = \sum_{i \in 	ext{all tenors}} \Delta P_{t,i} 	imes 	ext{True Sensi}_i
\]

\[
	ext{Diff}_t = 	ext{True PNL}_t - 	ext{PCA PNL}_t
\]

\[
	ext{Diff \%}_t = rac{	ext{Diff}_t}{	ext{True PNL}_t}
\]

When `True PNL = 0`, `Diff %` is set to `NaN`.



## How to use this notebook

1. Put this notebook in the same folder as:
   - `market_data.py`
   - `covariance_estimators.py`
   - `utils.py`
   - `pca_engines_with_pnl.py`
2. Update the **configuration cell**.
3. Run the notebook section by section.
4. Use the wide static study to identify a shortlist.
5. Use the rolling and regime sections to decide what is truly robust.

The notebook can work in two modes:

- **desk environment** with internal data access
- **file mode** using CSV / XLSX historical curve data

A lightweight compatibility shim is included so that `pyJade` is not required when you only work from files.


In [ ]:

# Compatibility shim for environments where pyJade is not installed.
# This lets the local modules import successfully when you work from files
# with explicit start_date / end_date.

from pathlib import Path
import sys
import types

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from pyJade.pearl import pearl_service  # noqa: F401
except ModuleNotFoundError:
    pyjade_module = types.ModuleType("pyJade")
    pearl_module = types.ModuleType("pyJade.pearl")

    def pearl_service():
        class DummyPearlService:
            def __getattr__(self, attribute_name):
                raise ModuleNotFoundError(
                    "pyJade is not available in this environment. "
                    "Database fetching and pearl-based date arithmetic are unavailable. "
                    "Use file mode with explicit start_date/end_date, or run this notebook on the desk environment."
                )
        return DummyPearlService()

    pearl_module.pearl_service = pearl_service
    pyjade_module.pearl = pearl_module

    sys.modules["pyJade"] = pyjade_module
    sys.modules["pyJade.pearl"] = pearl_module


In [ ]:

import itertools
import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from market_data import MarketData, CurveInfo
from covariance_estimators import CovarianceEstimator
from pca_engines_with_pnl import (
    StaticPCA,
    project_pca_loadings_on_hedge_tenors,
    compute_pca_hedge,
    compare_true_and_pca_pnl,
    backtest_pca_hedge,
)
from utils import reconstruction_metrics

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

warnings.filterwarnings("ignore")



## Configuration

This is the only cell you should need to edit first.

A few practical remarks:

- The notebook is much more informative when the **true sensitivity vector** is realistic.
- If you want to compare hedge robustness across books, rerun the hedge sections with several books.
- The combinatorics of hedge-tenor search can explode very quickly. Start with a controlled search universe.


In [ ]:

# =========================
# USER CONFIGURATION
# =========================

# -------- Data source --------
run_mode = "file"   # "file" or "db"
file_path = "your_rates_history.csv"   # used only in file mode

curve_info = CurveInfo(
    index="ESTER",
    ccy="EUR",
    curve_id="YOUR_CURVE_ID",
    closing="CLOSE",
    is_rate=True,
)

# Either explicit date range ...
start_date = "01/01/20"
end_date = "31/12/25"

# ... or desk-style period mode if you are in the internal environment
asofdate = None
period = None

data_source_name = "SUMMIT"  # used only in db mode

# -------- Universe --------
rates_tenors_universe = [
    "2Y", "3Y", "4Y", "5Y", "6Y", "7Y", "8Y", "9Y", "10Y", "11Y",
    "12Y", "13Y", "14Y", "15Y", "20Y", "25Y", "30Y", "40Y", "50Y"
]

# -------- Hedge book / sensitivity vector --------
# Replace this by your own book sensitivities.
true_sensi_position = pd.Series({
    "2Y": 0.0,
    "3Y": 0.0,
    "4Y": 0.0,
    "5Y": 125000.0,
    "6Y": 0.0,
    "7Y": -40000.0,
    "8Y": 0.0,
    "9Y": 0.0,
    "10Y": 210000.0,
    "11Y": 0.0,
    "12Y": 0.0,
    "13Y": 0.0,
    "14Y": 0.0,
    "15Y": -150000.0,
    "20Y": 90000.0,
    "25Y": 0.0,
    "30Y": -120000.0,
    "40Y": 0.0,
    "50Y": 0.0,
})

# -------- PNL backtest configuration --------
pnl_horizon_days = 5

# -------- Static grid --------
preprocessing_grid = [
    {"compute_changes": True,  "demean": False, "standardize": False},
    {"compute_changes": True,  "demean": True,  "standardize": False},
    {"compute_changes": True,  "demean": False, "standardize": True},
    {"compute_changes": False, "demean": True,  "standardize": False},
    {"compute_changes": False, "demean": False, "standardize": True},
]

covariance_grid = [
    {"method": "sample"},
    {"method": "ledoit_wolf"},
    {"method": "OAS"},
    {"method": "ewma", "halflife": 20},
    {"method": "ewma", "halflife": 60},
]

n_components_grid = [1, 2, 3, 4]

# Hedge-tenor search universe for combinatorial tests.
# Keep it smaller than the full curve if you want the notebook to remain practical.
hedge_search_universe = ["2Y", "5Y", "7Y", "10Y", "15Y", "20Y", "30Y", "50Y"]

# Manual hedge-tenor sets you definitely want to test.
manual_hedge_sets = {
    1: [("10Y",), ("15Y",), ("20Y",)],
    2: [("5Y", "10Y"), ("5Y", "30Y"), ("10Y", "30Y"), ("10Y", "20Y")],
    3: [("5Y", "10Y", "30Y"), ("5Y", "15Y", "30Y"), ("2Y", "10Y", "30Y")],
    4: [("2Y", "5Y", "10Y", "30Y")],
}

# Automatically add all combinations from the search universe up to this component count.
add_all_combinations_up_to_k = 3
max_hedge_combinations_per_k = 2000

# -------- Rolling studies --------
rolling_window_rows_grid = [126, 252, 504]   # roughly 6M / 1Y / 2Y in business-day rows
rolling_step_rows = 21                       # roughly 1M

# -------- Plot controls --------
plot_sample_tenors = ["2Y", "5Y", "10Y", "30Y", "50Y"]
number_of_best_models_to_display = 20
number_of_shortlisted_models_for_deep_dive = 5

# -------- Ranking weights (for convenience only, not for blind automatic selection) --------
ranking_weights = {
    "mean_abs_diff_ratio": 0.35,
    "rmse_diff": 0.20,
    "p95_abs_diff_pct": 0.15,
    "pnl_correlation_penalty": 0.15,
    "loading_condition_number": 0.05,
    "explained_var_penalty": 0.10,
}



## Helper functions

The functions below deliberately separate the problem into two blocks:

- **statistical PCA diagnostics**
- **hedge usefulness diagnostics**

That matters because a model can look statistically good while still being a poor hedge model.


In [ ]:

# =========================
# HELPER FUNCTIONS
# =========================

def make_base_market_data() -> MarketData:
    if run_mode == "file":
        market_data = MarketData(
            curve_info=curve_info,
            source=file_path,
            start_date=start_date,
            end_date=end_date,
            rates_tenors_universe=rates_tenors_universe,
        )
    elif run_mode == "db":
        market_data = MarketData(
            curve_info=curve_info,
            source=data_source_name,
            asofdate=asofdate,
            period=period,
            start_date=start_date if start_date and end_date else None,
            end_date=end_date if start_date and end_date else None,
            rates_tenors_universe=rates_tenors_universe,
        )
    else:
        raise ValueError("run_mode must be either 'file' or 'db'.")

    market_data.load()
    return market_data


def clone_market_data_with_preprocessing(
    raw_market_data: pd.DataFrame,
    preprocessing_spec: Dict,
) -> MarketData:
    if start_date is not None and end_date is not None:
        cloned_market_data = MarketData(
            curve_info=curve_info,
            source=file_path if run_mode == "file" else data_source_name,
            start_date=start_date,
            end_date=end_date,
            rates_tenors_universe=rates_tenors_universe,
        )
    else:
        cloned_market_data = MarketData(
            curve_info=curve_info,
            source=file_path if run_mode == "file" else data_source_name,
            asofdate=asofdate,
            period=period,
            rates_tenors_universe=rates_tenors_universe,
        )

    cloned_market_data.data = raw_market_data.copy()
    cloned_market_data.preprocess(**preprocessing_spec)
    return cloned_market_data


def build_covariance_estimator(covariance_spec: Dict) -> CovarianceEstimator:
    return CovarianceEstimator(**covariance_spec)


def preprocessing_label(preprocessing_spec: Dict) -> str:
    return (
        f"chg={int(preprocessing_spec['compute_changes'])} | "
        f"demean={int(preprocessing_spec['demean'])} | "
        f"std={int(preprocessing_spec['standardize'])}"
    )


def covariance_label(covariance_spec: Dict) -> str:
    method = covariance_spec["method"]
    if method == "ewma":
        halflife = covariance_spec.get("halflife", None)
        alpha_ewma = covariance_spec.get("alpha_ewma", None)
        if halflife is not None:
            return f"ewma_hl={halflife}"
        if alpha_ewma is not None:
            return f"ewma_alpha={alpha_ewma}"
    return str(method)


def fit_static_pca_model(
    raw_market_data: pd.DataFrame,
    preprocessing_spec: Dict,
    covariance_spec: Dict,
    n_components: int,
) -> Tuple[MarketData, StaticPCA]:
    processed_market_data = clone_market_data_with_preprocessing(
        raw_market_data=raw_market_data,
        preprocessing_spec=preprocessing_spec,
    )

    covariance_estimator = build_covariance_estimator(covariance_spec)

    pca_model = StaticPCA(
        data=processed_market_data,
        n_components=n_components,
        cov_estimator=covariance_estimator,
    )
    pca_model.fit()

    return processed_market_data, pca_model


def build_candidate_hedge_sets(
    n_components: int,
    hedge_universe: List[str],
    manual_sets_dictionary: Dict[int, List[Tuple[str, ...]]],
    add_all_combinations: bool = True,
    max_combinations: int = 2000,
) -> List[Tuple[str, ...]]:
    candidate_sets = []

    for hedge_tuple in manual_sets_dictionary.get(n_components, []):
        if len(hedge_tuple) == n_components:
            candidate_sets.append(tuple(hedge_tuple))

    if add_all_combinations:
        all_combinations = list(itertools.combinations(hedge_universe, n_components))
        if len(all_combinations) > max_combinations:
            raise ValueError(
                f"There are {len(all_combinations)} hedge combinations for k={n_components}, "
                f"which exceeds max_combinations={max_combinations}. Reduce hedge_search_universe."
            )
        candidate_sets.extend(all_combinations)

    # Deduplicate while preserving order
    deduplicated_sets = []
    seen = set()
    for hedge_tuple in candidate_sets:
        if hedge_tuple not in seen:
            deduplicated_sets.append(hedge_tuple)
            seen.add(hedge_tuple)

    return deduplicated_sets


def compute_loading_condition_number(
    pca_model: StaticPCA,
    hedge_tenors: Tuple[str, ...],
) -> float:
    loading_matrix = pca_model.results.loadings.loc[list(hedge_tenors)].values
    try:
        return float(np.linalg.cond(loading_matrix))
    except Exception:
        return np.nan


def summarize_pnl_quality(pnl_results) -> Dict[str, float]:
    pnl_comparison = pnl_results.comparison.copy()

    true_pnl = pnl_comparison["True PNL"]
    pca_pnl = pnl_comparison["PCA PNL"]
    pnl_diff = pnl_comparison["Diff"]
    pnl_diff_pct = pnl_comparison["Diff %"]

    mean_abs_true_pnl = true_pnl.abs().mean()
    mean_abs_diff = pnl_diff.abs().mean()
    rmse_diff = float(np.sqrt(np.mean(np.square(pnl_diff))))
    tracking_error = float(pnl_diff.std(ddof=1))
    pnl_correlation = float(true_pnl.corr(pca_pnl)) if len(pnl_comparison) >= 2 else np.nan

    true_pnl_variance = float(true_pnl.var(ddof=1))
    diff_variance = float(pnl_diff.var(ddof=1))
    hedge_effectiveness = np.nan
    if np.isfinite(true_pnl_variance) and true_pnl_variance > 0:
        hedge_effectiveness = 1.0 - diff_variance / true_pnl_variance

    sign_match_ratio = float((np.sign(true_pnl) == np.sign(pca_pnl)).mean())

    return {
        "mean_abs_true_pnl": float(mean_abs_true_pnl),
        "mean_abs_diff": float(mean_abs_diff),
        "mean_abs_diff_ratio": float(mean_abs_diff / mean_abs_true_pnl) if mean_abs_true_pnl != 0 else np.nan,
        "rmse_diff": rmse_diff,
        "tracking_error": tracking_error,
        "pnl_correlation": pnl_correlation,
        "hedge_effectiveness": float(hedge_effectiveness) if pd.notna(hedge_effectiveness) else np.nan,
        "mean_abs_diff_pct": float(pnl_diff_pct.abs().mean()),
        "median_abs_diff_pct": float(pnl_diff_pct.abs().median()),
        "p95_abs_diff_pct": float(pnl_diff_pct.abs().quantile(0.95)),
        "max_abs_diff_pct": float(pnl_diff_pct.abs().max()),
        "mean_signed_diff": float(pnl_diff.mean()),
        "sign_match_ratio": sign_match_ratio,
    }


def evaluate_single_static_configuration(
    raw_market_data: pd.DataFrame,
    preprocessing_spec: Dict,
    covariance_spec: Dict,
    n_components: int,
    candidate_hedge_sets: List[Tuple[str, ...]],
    true_sensitivity_vector: pd.Series,
    pnl_horizon_days: int,
):
    processed_market_data, pca_model = fit_static_pca_model(
        raw_market_data=raw_market_data,
        preprocessing_spec=preprocessing_spec,
        covariance_spec=covariance_spec,
        n_components=n_components,
    )

    reconstructed_levels, actual_levels, residual_levels = pca_model.reconstruct(revert_processing=True)
    reconstruction_summary = reconstruction_metrics(actual_levels, residual_levels)

    model_level_metrics = {
        "explained_var_topk": float(np.sum(pca_model.results.explained_var_ratios[:n_components])),
        "rmse_global": float(reconstruction_summary["RMSE Global"]),
        "mae_global": float(reconstruction_summary["MAE Global"]),
        "r2": float(reconstruction_summary["R2"]),
    }

    per_hedge_rows = []
    per_hedge_artifacts = {}

    for hedge_tenors in candidate_hedge_sets:
        if len(hedge_tenors) != n_components:
            continue

        try:
            pnl_results = backtest_pca_hedge(
                pca=pca_model,
                true_sensi_position=true_sensitivity_vector,
                hedge_tenors=list(hedge_tenors),
                horizon_days=pnl_horizon_days,
            )
            pnl_summary = summarize_pnl_quality(pnl_results)
            loading_condition_number = compute_loading_condition_number(pca_model, hedge_tenors)
        except Exception as exception_message:
            per_hedge_rows.append({
                "preprocessing_label": preprocessing_label(preprocessing_spec),
                "covariance_label": covariance_label(covariance_spec),
                "n_components": n_components,
                "hedge_tenors": hedge_tenors,
                "error": str(exception_message),
            })
            continue

        row = {
            "preprocessing_label": preprocessing_label(preprocessing_spec),
            "covariance_label": covariance_label(covariance_spec),
            "n_components": n_components,
            "hedge_tenors": hedge_tenors,
            "loading_condition_number": loading_condition_number,
            **model_level_metrics,
            **pnl_summary,
        }
        per_hedge_rows.append(row)

        model_identifier = (
            preprocessing_label(preprocessing_spec),
            covariance_label(covariance_spec),
            n_components,
            hedge_tenors,
        )
        per_hedge_artifacts[model_identifier] = {
            "processed_market_data": processed_market_data,
            "pca_model": pca_model,
            "pnl_results": pnl_results,
            "reconstructed_levels": reconstructed_levels,
            "actual_levels": actual_levels,
            "residual_levels": residual_levels,
        }

    return per_hedge_rows, per_hedge_artifacts


def add_convenience_ranking(static_results: pd.DataFrame) -> pd.DataFrame:
    ranked_results = static_results.copy()

    valid_rows = ranked_results.dropna(subset=[
        "mean_abs_diff_ratio",
        "rmse_diff",
        "p95_abs_diff_pct",
        "pnl_correlation",
        "loading_condition_number",
        "explained_var_topk",
    ]).copy()

    if valid_rows.empty:
        ranked_results["convenience_score"] = np.nan
        return ranked_results

    valid_rows["pnl_correlation_penalty"] = 1.0 - valid_rows["pnl_correlation"].clip(-1.0, 1.0)
    valid_rows["explained_var_penalty"] = 1.0 - valid_rows["explained_var_topk"].clip(0.0, 1.0)

    score_components = [
        "mean_abs_diff_ratio",
        "rmse_diff",
        "p95_abs_diff_pct",
        "pnl_correlation_penalty",
        "loading_condition_number",
        "explained_var_penalty",
    ]

    for metric_name in score_components:
        metric_series = valid_rows[metric_name].replace([np.inf, -np.inf], np.nan)
        lower_quantile = metric_series.quantile(0.05)
        upper_quantile = metric_series.quantile(0.95)
        clipped_series = metric_series.clip(lower=lower_quantile, upper=upper_quantile)
        denom = clipped_series.max() - clipped_series.min()
        if denom == 0 or not np.isfinite(denom):
            valid_rows[f"{metric_name}_normalized"] = 0.0
        else:
            valid_rows[f"{metric_name}_normalized"] = (clipped_series - clipped_series.min()) / denom

    valid_rows["convenience_score"] = 0.0
    for metric_name, metric_weight in ranking_weights.items():
        valid_rows["convenience_score"] += metric_weight * valid_rows[f"{metric_name}_normalized"]

    ranked_results = ranked_results.merge(
        valid_rows[["preprocessing_label", "covariance_label", "n_components", "hedge_tenors", "convenience_score"]],
        on=["preprocessing_label", "covariance_label", "n_components", "hedge_tenors"],
        how="left",
    )

    return ranked_results


def plot_curve_panel(raw_market_data: pd.DataFrame, selected_tenors: List[str]):
    figure, axis = plt.subplots(figsize=(14, 5))
    for tenor_name in selected_tenors:
        if tenor_name in raw_market_data.columns:
            axis.plot(raw_market_data.index, raw_market_data[tenor_name], label=tenor_name)
    axis.set_title("Raw market history")
    axis.set_ylabel("Level")
    axis.legend(ncol=min(len(selected_tenors), 5))
    axis.grid(True, alpha=0.3)
    plt.show()


def plot_daily_change_volatility(raw_market_data: pd.DataFrame):
    daily_changes = raw_market_data.diff().dropna()
    annualized_volatility = daily_changes.std() * np.sqrt(252)

    figure, axis = plt.subplots(figsize=(14, 5))
    annualized_volatility.plot(kind="bar", ax=axis)
    axis.set_title("Annualized volatility of daily changes by tenor")
    axis.set_ylabel("Volatility")
    axis.grid(True, axis="y", alpha=0.3)
    plt.show()


def plot_correlation_heatmap(raw_market_data: pd.DataFrame, use_changes: bool = True):
    data_for_corr = raw_market_data.diff().dropna() if use_changes else raw_market_data.copy()
    correlation_matrix = data_for_corr.corr()

    figure, axis = plt.subplots(figsize=(10, 8))
    heatmap = axis.imshow(correlation_matrix.values, aspect="auto")
    axis.set_title("Correlation matrix" + (" of daily changes" if use_changes else " of levels"))
    axis.set_xticks(range(len(correlation_matrix.columns)))
    axis.set_xticklabels(correlation_matrix.columns, rotation=90)
    axis.set_yticks(range(len(correlation_matrix.index)))
    axis.set_yticklabels(correlation_matrix.index)
    figure.colorbar(heatmap, ax=axis)
    plt.show()


def plot_metric_frontier(static_results: pd.DataFrame):
    valid_results = static_results.dropna(subset=["explained_var_topk", "mean_abs_diff_ratio", "n_components"])
    if valid_results.empty:
        print("No valid results to plot.")
        return

    figure, axis = plt.subplots(figsize=(10, 6))
    scatter = axis.scatter(
        valid_results["explained_var_topk"],
        valid_results["mean_abs_diff_ratio"],
        c=valid_results["n_components"],
        s=40,
        alpha=0.7,
    )
    axis.set_xlabel("Explained variance of retained PCs")
    axis.set_ylabel("Mean absolute hedge error / mean absolute true PNL")
    axis.set_title("Statistical fit versus hedge usefulness")
    axis.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=axis, label="n_components")
    plt.show()


def plot_best_model_pnl_diagnostics(pnl_results, title: str = ""):
    pnl_comparison = pnl_results.comparison.copy()

    figure, axes = plt.subplots(3, 1, figsize=(14, 14))

    axes[0].plot(pnl_comparison.index, pnl_comparison["True PNL"], label="True PNL")
    axes[0].plot(pnl_comparison.index, pnl_comparison["PCA PNL"], label="PCA PNL")
    axes[0].set_title(f"True PNL versus PCA PNL {title}")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(pnl_comparison.index, pnl_comparison["Diff"])
    axes[1].axhline(0.0, linestyle="--", linewidth=1.0)
    axes[1].set_title("Tracking error time series")
    axes[1].grid(True, alpha=0.3)

    axes[2].scatter(pnl_comparison["True PNL"], pnl_comparison["PCA PNL"], alpha=0.5)
    min_value = np.nanmin([pnl_comparison["True PNL"].min(), pnl_comparison["PCA PNL"].min()])
    max_value = np.nanmax([pnl_comparison["True PNL"].max(), pnl_comparison["PCA PNL"].max()])
    axes[2].plot([min_value, max_value], [min_value, max_value], linestyle="--")
    axes[2].set_xlabel("True PNL")
    axes[2].set_ylabel("PCA PNL")
    axes[2].set_title("Scatter: hedge replication quality")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_loadings(pca_model: StaticPCA, title: str = ""):
    loadings = pca_model.results.loadings.copy()
    figure, axis = plt.subplots(figsize=(14, 5))
    for column_name in loadings.columns:
        axis.plot(loadings.index, loadings[column_name], marker="o", label=column_name)
    axis.axhline(0.0, linestyle="--", linewidth=1.0)
    axis.set_title(f"PCA loadings {title}")
    axis.legend()
    axis.grid(True, alpha=0.3)
    plt.show()


def plot_explained_variance(pca_model: StaticPCA, title: str = ""):
    explained_variance = pca_model.results.explained_var_ratios[:pca_model.n_components]
    cumulative_explained_variance = np.cumsum(explained_variance)
    component_labels = [f"PC{i+1}" for i in range(len(explained_variance))]

    figure, axis = plt.subplots(figsize=(10, 5))
    axis.bar(component_labels, explained_variance)
    axis.plot(component_labels, cumulative_explained_variance, marker="o")
    axis.set_title(f"Explained variance {title}")
    axis.set_ylabel("Share of variance")
    axis.grid(True, axis="y", alpha=0.3)
    plt.show()


def plot_reconstruction_example(actual_levels: pd.DataFrame, reconstructed_levels: pd.DataFrame, selected_tenors: List[str]):
    figure, axis = plt.subplots(figsize=(14, 5))
    for tenor_name in selected_tenors:
        if tenor_name not in actual_levels.columns:
            continue
        axis.plot(actual_levels.index, actual_levels[tenor_name], label=f"{tenor_name} actual")
        axis.plot(reconstructed_levels.index, reconstructed_levels[tenor_name], linestyle="--", label=f"{tenor_name} reconstructed")
    axis.set_title("Reconstruction on selected tenors")
    axis.legend(ncol=2)
    axis.grid(True, alpha=0.3)
    plt.show()


def assign_market_regimes_from_price_moves(price_moves: pd.DataFrame) -> pd.Series:
    stress_indicator = price_moves.abs().mean(axis=1)
    return pd.qcut(
        stress_indicator.rank(method="first"),
        q=4,
        labels=["calm", "normal", "active", "stressed"],
    )


def regime_breakdown_table(pnl_results) -> pd.DataFrame:
    pnl_comparison = pnl_results.comparison.copy()
    regimes = assign_market_regimes_from_price_moves(pnl_results.price_moves)
    pnl_comparison["regime"] = regimes

    rows = []
    for regime_name, regime_frame in pnl_comparison.groupby("regime"):
        rows.append({
            "regime": regime_name,
            "n_obs": int(len(regime_frame)),
            "mean_abs_diff": float(regime_frame["Diff"].abs().mean()),
            "rmse_diff": float(np.sqrt(np.mean(np.square(regime_frame["Diff"])))),
            "mean_abs_diff_pct": float(regime_frame["Diff %"].abs().mean()),
            "pnl_corr": float(regime_frame["True PNL"].corr(regime_frame["PCA PNL"])),
        })
    return pd.DataFrame(rows).sort_values("regime")


In [ ]:

# =========================
# ROLLING STUDY FUNCTIONS
# =========================

def component_similarity_matrix(reference_loadings: pd.DataFrame, candidate_loadings: pd.DataFrame) -> np.ndarray:
    reference_matrix = reference_loadings.values
    candidate_matrix = candidate_loadings.values

    reference_norms = np.linalg.norm(reference_matrix, axis=0, keepdims=True)
    candidate_norms = np.linalg.norm(candidate_matrix, axis=0, keepdims=True)

    reference_unit = reference_matrix / reference_norms
    candidate_unit = candidate_matrix / candidate_norms

    return reference_unit.T @ candidate_unit


def best_permutation_from_similarity(similarity_matrix: np.ndarray) -> Tuple[int, ...]:
    n_components = similarity_matrix.shape[0]
    best_permutation = None
    best_score = -np.inf
    for permutation in itertools.permutations(range(n_components)):
        score = sum(abs(similarity_matrix[row_index, permutation[row_index]]) for row_index in range(n_components))
        if score > best_score:
            best_score = score
            best_permutation = permutation
    return tuple(best_permutation)


def align_loadings_to_reference(reference_loadings: pd.DataFrame, current_loadings: pd.DataFrame) -> pd.DataFrame:
    similarity_matrix = component_similarity_matrix(reference_loadings, current_loadings)
    permutation = best_permutation_from_similarity(similarity_matrix)
    aligned_loadings = current_loadings.iloc[:, list(permutation)].copy()
    aligned_loadings.columns = reference_loadings.columns

    for component_name in reference_loadings.columns:
        signed_dot = float(np.dot(reference_loadings[component_name].values, aligned_loadings[component_name].values))
        if signed_dot < 0:
            aligned_loadings[component_name] *= -1.0

    return aligned_loadings


def fit_pca_on_processed_window(
    processed_window: pd.DataFrame,
    raw_window: pd.DataFrame,
    covariance_spec: Dict,
    n_components: int,
) -> StaticPCA:
    temporary_market_data = clone_market_data_with_preprocessing(
        raw_market_data=raw_window,
        preprocessing_spec={"compute_changes": False, "demean": False, "standardize": False},
    )
    temporary_market_data.processed_data = processed_window.copy()
    temporary_market_data.data = raw_window.copy()

    covariance_estimator = build_covariance_estimator(covariance_spec)
    pca_model = StaticPCA(
        data=temporary_market_data,
        n_components=n_components,
        cov_estimator=covariance_estimator,
    )
    pca_model.fit()
    return pca_model


def run_rolling_loading_study(
    raw_market_data: pd.DataFrame,
    preprocessing_spec: Dict,
    covariance_spec: Dict,
    n_components: int,
    window_rows: int,
    step_rows: int,
) -> pd.DataFrame:
    processed_market_data = clone_market_data_with_preprocessing(raw_market_data, preprocessing_spec)
    processed_data = processed_market_data.processed_data.copy()
    raw_data = processed_market_data.data.copy().loc[processed_data.index.min():processed_data.index.max()]

    rows = []
    previous_loadings = None

    for end_row in range(window_rows, len(processed_data) + 1, step_rows):
        processed_window = processed_data.iloc[end_row - window_rows:end_row].copy()
        raw_window = raw_data.loc[processed_window.index.min():processed_window.index.max()].copy()

        pca_model = fit_pca_on_processed_window(
            processed_window=processed_window,
            raw_window=raw_window,
            covariance_spec=covariance_spec,
            n_components=n_components,
        )

        current_loadings = pca_model.results.loadings.copy()
        if previous_loadings is not None:
            current_loadings = align_loadings_to_reference(previous_loadings, current_loadings)

        row = {
            "window_end": processed_window.index[-1],
            "explained_var_topk": float(np.sum(pca_model.results.explained_var_ratios[:n_components])),
        }

        for component_index in range(n_components):
            row[f"PC{component_index + 1}_explained_var"] = float(pca_model.results.explained_var_ratios[component_index])
            if previous_loadings is not None:
                previous_vector = previous_loadings.iloc[:, component_index].values
                current_vector = current_loadings.iloc[:, component_index].values
                cosine_similarity = float(
                    np.dot(previous_vector, current_vector) /
                    (np.linalg.norm(previous_vector) * np.linalg.norm(current_vector))
                )
                row[f"PC{component_index + 1}_cosine_similarity"] = cosine_similarity
            else:
                row[f"PC{component_index + 1}_cosine_similarity"] = np.nan

        rows.append(row)
        previous_loadings = current_loadings.copy()

    return pd.DataFrame(rows).set_index("window_end")


def run_rolling_out_of_sample_pnl_study(
    raw_market_data: pd.DataFrame,
    preprocessing_spec: Dict,
    covariance_spec: Dict,
    n_components: int,
    hedge_tenors: Tuple[str, ...],
    true_sensitivity_vector: pd.Series,
    pnl_horizon_days: int,
    window_rows: int,
    step_rows: int,
) -> pd.DataFrame:
    processed_market_data = clone_market_data_with_preprocessing(raw_market_data, preprocessing_spec)
    processed_data = processed_market_data.processed_data.copy()
    raw_data = processed_market_data.data.copy().sort_index()

    rolling_pnl_blocks = []

    for end_row in range(window_rows, len(processed_data) - step_rows + 1, step_rows):
        processed_train = processed_data.iloc[end_row - window_rows:end_row].copy()
        raw_train = raw_data.loc[processed_train.index.min():processed_train.index.max()].copy()

        pca_model = fit_pca_on_processed_window(
            processed_window=processed_train,
            raw_window=raw_train,
            covariance_spec=covariance_spec,
            n_components=n_components,
        )

        projection_matrix = project_pca_loadings_on_hedge_tenors(
            loadings=pca_model.results.loadings,
            hedge_tenors=list(hedge_tenors),
        )
        pca_sensitivity = compute_pca_hedge(
            sensi=true_sensitivity_vector,
            proj_matrix=projection_matrix,
        )

        test_start_index = end_row
        test_end_index = min(end_row + step_rows + pnl_horizon_days, len(raw_data))
        raw_test_slice = raw_data.iloc[test_start_index:test_end_index].copy()
        if len(raw_test_slice) <= pnl_horizon_days:
            continue

        pnl_results = compare_true_and_pca_pnl(
            market_data=raw_test_slice,
            true_sensi_position=true_sensitivity_vector,
            pca_sensi=pca_sensitivity,
            horizon_days=pnl_horizon_days,
        )
        pnl_frame = pnl_results.comparison.copy().iloc[:step_rows].copy()
        pnl_frame["refit_date"] = processed_train.index[-1]
        rolling_pnl_blocks.append(pnl_frame)

    if not rolling_pnl_blocks:
        return pd.DataFrame()

    return pd.concat(rolling_pnl_blocks).sort_index()


def summarize_rolling_out_of_sample_pnl(rolling_pnl_frame: pd.DataFrame) -> pd.Series:
    if rolling_pnl_frame.empty:
        return pd.Series(dtype=float)

    true_pnl = rolling_pnl_frame["True PNL"]
    pca_pnl = rolling_pnl_frame["PCA PNL"]
    pnl_diff = rolling_pnl_frame["Diff"]
    pnl_diff_pct = rolling_pnl_frame["Diff %"]

    return pd.Series({
        "mean_abs_diff": float(pnl_diff.abs().mean()),
        "rmse_diff": float(np.sqrt(np.mean(np.square(pnl_diff)))),
        "mean_abs_diff_pct": float(pnl_diff_pct.abs().mean()),
        "p95_abs_diff_pct": float(pnl_diff_pct.abs().quantile(0.95)),
        "pnl_correlation": float(true_pnl.corr(pca_pnl)),
        "sign_match_ratio": float((np.sign(true_pnl) == np.sign(pca_pnl)).mean()),
    })


def plot_rolling_loading_study(rolling_loading_frame: pd.DataFrame, title: str = ""):
    if rolling_loading_frame.empty:
        print("No rolling loading results to plot.")
        return

    figure, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    similarity_columns = [column_name for column_name in rolling_loading_frame.columns if column_name.endswith("cosine_similarity")]
    explained_var_columns = [column_name for column_name in rolling_loading_frame.columns if column_name.endswith("explained_var") and column_name != "explained_var_topk"]

    rolling_loading_frame[similarity_columns].plot(ax=axes[0])
    axes[0].set_title(f"Rolling loading stability {title}")
    axes[0].set_ylabel("Cosine similarity")
    axes[0].grid(True, alpha=0.3)

    rolling_loading_frame[explained_var_columns + ["explained_var_topk"]].plot(ax=axes[1])
    axes[1].set_title("Explained variance through time")
    axes[1].set_ylabel("Variance share")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()



## Load the market data and do a first sanity check

Before comparing models, always verify:

- no missing tenor in the universe
- enough history
- reasonable date ordering
- no obvious discontinuity / file parsing issue


In [ ]:

base_market_data = make_base_market_data()
raw_market_data = base_market_data.data.copy().sort_index()

print(f"Raw data shape: {raw_market_data.shape}")
print(f"Date range: {raw_market_data.index.min()} -> {raw_market_data.index.max()}")
display(raw_market_data.head())
display(raw_market_data.tail())


In [ ]:

plot_curve_panel(raw_market_data, plot_sample_tenors)
plot_daily_change_volatility(raw_market_data)
plot_correlation_heatmap(raw_market_data, use_changes=True)



## Run the broad static study

This is the main grid that compares:

- preprocessing
- covariance estimator
- number of PCs
- hedge-tenor combinations

The output is a ranking table, not a blind answer.

A model should usually not be selected because it is first in one scalar score. It should survive the later diagnostics as well.


In [ ]:

all_static_rows = []
all_static_artifacts = {}

for preprocessing_spec in preprocessing_grid:
    for covariance_spec in covariance_grid:
        for n_components in n_components_grid:
            add_all_combinations = n_components <= add_all_combinations_up_to_k
            candidate_hedge_sets = build_candidate_hedge_sets(
                n_components=n_components,
                hedge_universe=hedge_search_universe,
                manual_sets_dictionary=manual_hedge_sets,
                add_all_combinations=add_all_combinations,
                max_combinations=max_hedge_combinations_per_k,
            )

            print(
                "Running static study | ",
                preprocessing_label(preprocessing_spec), " | ",
                covariance_label(covariance_spec), " | ",
                f"k={n_components}", " | ",
                f"hedge sets={len(candidate_hedge_sets)}"
            )

            static_rows, static_artifacts = evaluate_single_static_configuration(
                raw_market_data=raw_market_data,
                preprocessing_spec=preprocessing_spec,
                covariance_spec=covariance_spec,
                n_components=n_components,
                candidate_hedge_sets=candidate_hedge_sets,
                true_sensitivity_vector=true_sensi_position,
                pnl_horizon_days=pnl_horizon_days,
            )
            all_static_rows.extend(static_rows)
            all_static_artifacts.update(static_artifacts)

static_results = pd.DataFrame(all_static_rows)
static_results = add_convenience_ranking(static_results)

print(f"Number of tested configurations: {len(static_results)}")
display(static_results.head())


In [ ]:

static_results_clean = static_results.dropna(subset=["mean_abs_diff_ratio", "pnl_correlation"]).copy()
static_results_ranked = static_results_clean.sort_values(
    ["convenience_score", "mean_abs_diff_ratio", "pnl_correlation"],
    ascending=[True, True, False],
)

display(
    static_results_ranked[
        [
            "preprocessing_label",
            "covariance_label",
            "n_components",
            "hedge_tenors",
            "explained_var_topk",
            "rmse_global",
            "r2",
            "mean_abs_diff_ratio",
            "rmse_diff",
            "p95_abs_diff_pct",
            "pnl_correlation",
            "hedge_effectiveness",
            "loading_condition_number",
            "convenience_score",
        ]
    ].head(number_of_best_models_to_display)
)


In [ ]:

plot_metric_frontier(static_results_ranked)

summary_by_components = (
    static_results_ranked
    .groupby("n_components")
    [["explained_var_topk", "mean_abs_diff_ratio", "pnl_correlation", "loading_condition_number"]]
    .median()
)

summary_by_preprocessing = (
    static_results_ranked
    .groupby("preprocessing_label")
    [["explained_var_topk", "mean_abs_diff_ratio", "pnl_correlation", "loading_condition_number"]]
    .median()
    .sort_values("mean_abs_diff_ratio")
)

summary_by_covariance = (
    static_results_ranked
    .groupby("covariance_label")
    [["explained_var_topk", "mean_abs_diff_ratio", "pnl_correlation", "loading_condition_number"]]
    .median()
    .sort_values("mean_abs_diff_ratio")
)

print("Median metrics by number of components")
display(summary_by_components)

print("Median metrics by preprocessing choice")
display(summary_by_preprocessing)

print("Median metrics by covariance estimator")
display(summary_by_covariance)



## Shortlist and deep dive the best candidates

This section is where the notebook becomes useful for judgment.

A good shortlist usually contains models that:

- are good on hedge error metrics
- have reasonable explained variance
- do not rely on an ill-conditioned hedge mapping
- are intuitively interpretable


In [ ]:

shortlist = static_results_ranked.head(number_of_shortlisted_models_for_deep_dive).copy()
display(shortlist)


In [ ]:

for _, shortlisted_row in shortlist.iterrows():
    model_key = (
        shortlisted_row["preprocessing_label"],
        shortlisted_row["covariance_label"],
        int(shortlisted_row["n_components"]),
        tuple(shortlisted_row["hedge_tenors"]),
    )
    model_artifact = all_static_artifacts[model_key]
    pca_model = model_artifact["pca_model"]
    pnl_results = model_artifact["pnl_results"]
    actual_levels = model_artifact["actual_levels"]
    reconstructed_levels = model_artifact["reconstructed_levels"]

    title = (
        f"| {shortlisted_row['preprocessing_label']} | "
        f"{shortlisted_row['covariance_label']} | "
        f"k={int(shortlisted_row['n_components'])} | "
        f"hedge={tuple(shortlisted_row['hedge_tenors'])}"
    )

    plot_explained_variance(pca_model, title=title)
    plot_loadings(pca_model, title=title)
    plot_reconstruction_example(actual_levels, reconstructed_levels, plot_sample_tenors)
    plot_best_model_pnl_diagnostics(pnl_results, title=title)

    print("PCA hedge sensitivities")
    display(pnl_results.pca_sensi.to_frame("pca_sensi"))

    print("Regime breakdown")
    display(regime_breakdown_table(pnl_results))



## Suggested interpretation framework

At this point, you should usually answer the following:

1. Does going from `k=2` to `k=3` materially reduce hedge error?
2. Does going from `k=3` to `k=4` mostly add noise / instability rather than hedge benefit?
3. Is standardization helping because the long end dominates raw variance, or is it destroying desk intuition?
4. Is EWMA better because the market regime changes fast, or is it just overfitting recent noise?
5. Are the best hedge pillars economically intuitive and liquid enough for implementation?
6. Is a lower-statistical-fit model actually better in out-of-sample hedge replication?



## Rolling stability study

Static fit is not enough for your project because you want a hedge representation that remains meaningful when the market regime changes.

This section studies two things:

- **loading stability**: do the PCs remain similar through time?
- **out-of-sample hedge performance**: if you refit every month, does the hedge still work after the refit date?


In [ ]:

# Select which shortlisted models you want to run in rolling mode.
# By default, the notebook uses the current shortlist.
rolling_candidates = shortlist.copy()

rolling_summary_rows = []
rolling_detailed_results = {}

for _, candidate_row in rolling_candidates.iterrows():
    preprocessing_label_value = candidate_row["preprocessing_label"]
    covariance_label_value = candidate_row["covariance_label"]
    n_components_value = int(candidate_row["n_components"])
    hedge_tenors_value = tuple(candidate_row["hedge_tenors"])

    preprocessing_spec = next(spec for spec in preprocessing_grid if preprocessing_label(spec) == preprocessing_label_value)
    covariance_spec = next(spec for spec in covariance_grid if covariance_label(spec) == covariance_label_value)

    for window_rows in rolling_window_rows_grid:
        print(
            "Rolling study | ",
            preprocessing_label_value, " | ",
            covariance_label_value, " | ",
            f"k={n_components_value}", " | ",
            f"hedge={hedge_tenors_value}", " | ",
            f"window_rows={window_rows}"
        )

        rolling_loading_frame = run_rolling_loading_study(
            raw_market_data=raw_market_data,
            preprocessing_spec=preprocessing_spec,
            covariance_spec=covariance_spec,
            n_components=n_components_value,
            window_rows=window_rows,
            step_rows=rolling_step_rows,
        )

        rolling_pnl_frame = run_rolling_out_of_sample_pnl_study(
            raw_market_data=raw_market_data,
            preprocessing_spec=preprocessing_spec,
            covariance_spec=covariance_spec,
            n_components=n_components_value,
            hedge_tenors=hedge_tenors_value,
            true_sensitivity_vector=true_sensi_position,
            pnl_horizon_days=pnl_horizon_days,
            window_rows=window_rows,
            step_rows=rolling_step_rows,
        )

        rolling_pnl_summary = summarize_rolling_out_of_sample_pnl(rolling_pnl_frame)
        mean_loading_similarity = np.nanmean([
            rolling_loading_frame[column_name].mean()
            for column_name in rolling_loading_frame.columns
            if column_name.endswith("cosine_similarity")
        ]) if not rolling_loading_frame.empty else np.nan

        rolling_summary_row = {
            "preprocessing_label": preprocessing_label_value,
            "covariance_label": covariance_label_value,
            "n_components": n_components_value,
            "hedge_tenors": hedge_tenors_value,
            "window_rows": window_rows,
            "mean_loading_similarity": float(mean_loading_similarity) if pd.notna(mean_loading_similarity) else np.nan,
            **rolling_pnl_summary.to_dict(),
        }
        rolling_summary_rows.append(rolling_summary_row)

        rolling_key = (
            preprocessing_label_value,
            covariance_label_value,
            n_components_value,
            hedge_tenors_value,
            window_rows,
        )
        rolling_detailed_results[rolling_key] = {
            "rolling_loading_frame": rolling_loading_frame,
            "rolling_pnl_frame": rolling_pnl_frame,
        }

rolling_summary = pd.DataFrame(rolling_summary_rows)
rolling_summary = rolling_summary.sort_values(["rmse_diff", "mean_abs_diff_pct", "mean_loading_similarity"], ascending=[True, True, False])
display(rolling_summary)


In [ ]:

for rolling_key, rolling_payload in rolling_detailed_results.items():
    print("=" * 120)
    print(rolling_key)

    rolling_loading_frame = rolling_payload["rolling_loading_frame"]
    rolling_pnl_frame = rolling_payload["rolling_pnl_frame"]

    plot_rolling_loading_study(rolling_loading_frame, title=str(rolling_key))

    if not rolling_pnl_frame.empty:
        figure, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
        axes[0].plot(rolling_pnl_frame.index, rolling_pnl_frame["True PNL"], label="True PNL")
        axes[0].plot(rolling_pnl_frame.index, rolling_pnl_frame["PCA PNL"], label="PCA PNL")
        axes[0].set_title("Rolling out-of-sample PNL replication")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(rolling_pnl_frame.index, rolling_pnl_frame["Diff"])
        axes[1].axhline(0.0, linestyle="--", linewidth=1.0)
        axes[1].set_title("Rolling out-of-sample tracking error")
        axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        display(rolling_pnl_frame.head())



## Final decision template

Use the final decision as a structured desk note, for example:

### 1. Statistical conclusion
- best preprocessing choice
- best covariance estimator
- useful number of components before diminishing returns

### 2. Hedge conclusion
- best hedge-tenor set for the chosen `k`
- expected mean / tail hedge error
- stability of the hedge mapping

### 3. Robustness conclusion
- behavior in calm vs stressed regimes
- rolling loading stability
- rolling out-of-sample hedge replication quality

### 4. Implementation conclusion
- whether the model is simple enough to explain and trade
- whether the selected hedge tenors are liquid / practical
- whether you prefer a slightly worse in-sample model for a more stable out-of-sample hedge



## Extensions that are worth adding later

If you want to take the framework one step further for the project, the most useful additions would be:

1. **historical panel of books** instead of a single sensitivity vector
2. **rolling re-estimation of hedge notionals with transaction cost proxy**
3. **constraint handling** on hedge tenors or sign / liquidity buckets
4. **comparison with your implied sparse PCA regression framework**
5. **stress-event diagnostics** on specific dates rather than only unconditional averages

That is usually where the model selection becomes desk-grade rather than purely academic.
